# Step 6 — BAGEL pruned heterogeneous-batching correctness gate

This compares the same two prompt/seed requests under **B=1** and heterogeneous **B=2**. Both arms use the exact `drop_image` optimization established in Steps 4–5 (`cfg_text_scale=4`, `cfg_img_scale=1`, interval `[0.4, 1.0]`). It validates all 49 refinement steps, the two prepared CFG rows at every step, mixed-age B=2 execution, five saved latent boundaries, and final images.

This is a correctness gate, not a performance run: latent capture synchronizes selected steps. The notebook will not launch a repeated performance sweep. Expected wall time on an A100 is approximately **35–50 minutes**, including two independent server starts.

## 1. Mount Drive and create a local run directory

Active logs, traces, images, and the checkpoint cache stay on `/content`. Drive receives one verified ZIP only after both servers have stopped.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')
RUN_ID = 'pruned_batch_gate_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
RUN_COMMIT = 'c7ed2adb0fc7b019e6539afcd40c1b8ed7b9d95f'
REPO = Path('/content') / f'mstar_{RUN_ID}'
LOCAL_ROOT = Path('/content/bagel_runs')
RESULT_DIR = LOCAL_ROOT / RUN_ID
GATE_DIR = RESULT_DIR / 'drop_image_b1_vs_b2_gate'
CACHE_DIR = Path('/content/hf_cache_bagel')
UV_CACHE_DIR = Path('/content/uv_cache')
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/CMU_11868_BAGEL')
for path in (LOCAL_ROOT, RESULT_DIR, CACHE_DIR, UV_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=path != RESULT_DIR)
os.environ['UV_CACHE_DIR'] = str(UV_CACHE_DIR)
print({'run_id': RUN_ID, 'active_results': str(RESULT_DIR), 'drive_backup': str(DRIVE_BACKUP_DIR)})

## 2. Clone the pinned implementation and verify hardware

This stops before downloads if the runtime does not have an 80 GB-class GPU, enough host RAM, or enough local disk.

In [ ]:
subprocess.run(['git', 'clone', '--no-checkout', FORK_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', RUN_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == RUN_COMMIT, (head, RUN_COMMIT)
subprocess.run([
    sys.executable, str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'), '--disk-path', '/content',
    '--require-gpu-memory-gib', '75', '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '55',
], check=True)
source = {'fork_url': FORK_URL, 'requested_commit': RUN_COMMIT, 'resolved_commit': head,
          'git_status': subprocess.check_output(['git', 'status', '--porcelain=v1'], cwd=REPO, text=True)}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
print('Hardware preflight passed at', head)

## 3. Create or resume the pinned virtual environment

This cell is re-runnable within the same Colab VM. It does not place the environment on Drive.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output([sys.executable, '-c', 'import shutil; print(shutil.which("uv"))'], text=True).strip()
VENV = REPO / '.venv-bagel'
VENV_PY = VENV / 'bin/python'
if not VENV_PY.exists():
    subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
                '--constraint', str(constraints), '-e', f'{REPO}[bagel]', 'matplotlib==3.10.7'], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['PATH'] = os.pathsep.join((str(VENV / 'bin'), env['PATH']))
tests = ('test_bagel_image_batch_packing.py', 'test_execution_trace.py',
         'test_bagel_experiment_runtime.py', 'test_bagel_pruned_batch_gate.py')
for test in tests:
    subprocess.run([str(VENV_PY), str(REPO / 'test/modular' / test), '-v'], cwd=REPO, env=env, check=True)
gpu_check = subprocess.check_output([str(VENV_PY), '-c',
    'import torch; assert torch.cuda.is_available(); x=torch.ones((16,16),device="cuda",dtype=torch.bfloat16); y=x@x; torch.cuda.synchronize(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), float(y[0,0]))'],
    text=True, env=env)
(RESULT_DIR / 'gpu_import_check.txt').write_text(gpu_check)
print('Environment preflight passed:', gpu_check.strip())

## 4. Reuse or download the pinned BAGEL checkpoint locally

A second run in the same VM reuses `/content/hf_cache_bagel`. Every safetensors header is validated before a server starts.

In [ ]:
manifest_path = RESULT_DIR / 'checkpoint_manifest.json'
subprocess.run([str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
                '--cache-dir', str(CACHE_DIR), '--manifest', str(manifest_path)], cwd=REPO, env=env, check=True)
manifest = json.loads(manifest_path.read_text())
print('Validated checkpoint:', manifest['resolved_revision'])

## 5. Run only the B=1 versus B=2 correctness gate

The cell prints a heartbeat at least every 20 seconds. On failure or interruption it stops owned server children, packages all evidence locally, and then attempts a checksum-verified Drive backup. Do not rerun this cell against the same `RUN_ID`.

In [ ]:
sys.path.insert(0, str(REPO / 'experiments/bagel/scripts'))
from experiment_runtime import package_results, run_logged
command = [str(VENV_PY), str(REPO / 'experiments/bagel/scripts/run_pruned_batch_gate.py'),
           '--repo-root', str(REPO), '--cache-dir', str(CACHE_DIR),
           '--output-dir', str(GATE_DIR), '--port', '8026']
run_error = None
try:
    run_logged(command, RESULT_DIR / 'gate_console.log', cwd=REPO, env=env,
               stage='complete Step 6 correctness gate', timeout=7200)
except BaseException as exc:
    run_error = exc
finally:
    archive_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_results.zip'
    if archive_path.exists():
        print('Preserving existing local ZIP:', archive_path)
    else:
        receipt = package_results(RESULT_DIR, DRIVE_BACKUP_DIR)
        (RESULT_DIR.parent / f'{RESULT_DIR.name}_backup_receipt.json').write_text(json.dumps(receipt, indent=2) + '\n')
        print(json.dumps(receipt, indent=2))
if run_error is not None:
    raise run_error
print('Correctness gate completed:', GATE_DIR)

## 6. Inspect the gate and download the verified local ZIP

`exact_at_checked_boundaries` means exact saved latents at steps 0, 20, 40, 41, and 48 plus byte-identical final PNGs. Any other successful status is deliberately reported as `numerical_review_required`; this notebook does not invent an acceptance tolerance.

In [ ]:
report = json.loads((GATE_DIR / 'pruned_batch_gate.json').read_text())
print(json.dumps({
    'status': report['status'],
    'performance_authorized': report['performance_authorized'],
    'serial_trace': report['serial_trace'],
    'batched_trace': report['batched_trace'],
    'request_comparisons': report['requests'],
}, indent=2))
archive_path = RESULT_DIR.parent / f'{RESULT_DIR.name}_results.zip'
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Verified ZIP:', archive_path, 'SHA-256:', digest)
from google.colab import files
files.download(str(archive_path))

After the browser download completes, share the ZIP and printed report. Then use **Runtime → Disconnect and delete runtime**. We will review numerical drift before preparing the separate repeated performance experiment.